In [16]:
# ==========================================
# TrendPulse Part 4: Export SQL Results to CSV
# ==========================================

import sqlite3
import pandas as pd
import os

# 1. Connect to your SQLite database
conn = sqlite3.connect("trendpulse.db")

# 2. Run your global category query
global_category_query = """
    WITH all_videos AS (
        SELECT category_id, views, likes, country FROM videos_us
        UNION ALL
        SELECT category_id, views, likes, country FROM videos_gb
        UNION ALL
        SELECT category_id, views, likes, country FROM videos_in
    )
    SELECT
        c.category_name,
        COUNT(v.category_id) AS total_videos,
        ROUND(AVG(v.views), 2) AS avg_views,
        ROUND(AVG(v.likes), 2) AS avg_likes
    FROM all_videos v
    JOIN categories c ON CAST(v.category_id AS TEXT) = c.category_id
    GROUP BY c.category_name
    ORDER BY avg_views DESC;
"""

df_categories = pd.read_sql(global_category_query, conn)
conn.close()

# 3. Ensure the output directory exists
output_dir = "part4_sql_database/outputs"
os.makedirs(output_dir, exist_ok=True)

# 4. Define file path and export to CSV
output_path = os.path.join(output_dir, "global_category_performance.csv")
df_categories.to_csv(output_path, index=False)

print(f"✅ Successfully exported query results to: {output_path}")

# Display the head to verify
display(df_categories.head())

✅ Successfully exported query results to: part4_sql_database/outputs/global_category_performance.csv


,category_name,total_videos,avg_views,avg_likes
0,Music,10363,3187732.88,134776.23
1,Entertainment,12332,900146.61,28905.80
2,Comedy,3273,603166.89,28720.17
3,Film & Animation,2308,577611.68,17234.52
4,People & Blogs,3820,549958.24,19782.74


In [22]:
import os
import sqlite3
import pandas as pd

# 1. Ensure the part4 directory exists locally in Colab
os.makedirs("trendpulse/part4_sql_database/outputs", exist_ok=True)

# 2. Connect and save the database directly inside the part4 folder
db_path = "trendpulse/part4_sql_database/trendpulse.db"
conn = sqlite3.connect(db_path)

# (Your ingestion and queries run here, writing tables to db_path)

# 3. Export your CSV report directly into the outputs folder
output_path = "trendpulse/part4_sql_database/outputs/global_category_performance.csv"
# df_categories.to_csv(output_path, index=False)

conn.close()
print(f"✅ Database saved at: {db_path}")
print(f"✅ CSV report saved at: {output_path}")

✅ Database saved at: trendpulse/part4_sql_database/trendpulse.db
✅ CSV report saved at: trendpulse/part4_sql_database/outputs/global_category_performance.csv


In [23]:
import os
import sqlite3
import pandas as pd
import openpyxl
from openpyxl.formatting.rule import CellIsRule
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter
from openpyxl.workbook.defined_name import DefinedName

# 1. Load data from SQLite database
conn = sqlite3.connect("trendpulse.db")

df_us = pd.read_sql("SELECT * FROM videos_us", conn)
df_gb = pd.read_sql("SELECT * FROM videos_gb", conn)
df_in = pd.read_sql("SELECT * FROM videos_in", conn)
df_cat = pd.read_sql("SELECT * FROM categories", conn)

conn.close()

# Combine records into a single dataframe
df_records = pd.concat([df_us, df_gb, df_in], ignore_index=True)

# Define output path
output_dir = "trendpulse/part4_sql_database/outputs"
os.makedirs(output_dir, exist_ok=True)
excel_path = os.path.join(output_dir, "TrendPulse_Analysis.xlsx")

# 2. Write base DataFrames into Excel sheets
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_records.to_excel(writer, sheet_name="Records", index=False)
    df_cat.to_excel(writer, sheet_name="Categories_Lookup", index=False)

# 3. Enhance workbook with formulas, formatting, lookups, and pivot
wb = openpyxl.load_workbook(excel_path)
ws_records = wb["Records"]
ws_cat = wb["Categories_Lookup"]

# --- Requirement (f): Named Range ---
cat_max_row = ws_cat.max_row
cat_range = DefinedName("CatLookup", attr_text=f"Categories_Lookup!$A$2:$B${cat_max_row}")
wb.defined_names["CatLookup"] = cat_range

# --- Requirement (b): Lookup Sheet & Formula (VLOOKUP) ---
cat_id_col = None
for col in range(1, ws_records.max_column + 1):
    if ws_records.cell(row=1, column=col).value == "category_id":
        cat_id_col = get_column_letter(col)
        break

name_col_idx = ws_records.max_column + 1
ws_records.cell(row=1, column=name_col_idx, value="category_name")

for r in range(2, ws_records.max_row + 1):
    ws_records.cell(row=r, column=name_col_idx, value=f'=VLOOKUP({cat_id_col}{r}, CatLookup, 2, FALSE)')

# --- Requirement (c): Headline Formulas (COUNTIF, SUM, AVERAGE) ---
ws_summary = wb.create_sheet(title="Headline_Summary")
ws_summary["A1"] = "Headline Metric"
ws_summary["B1"] = "Formula Output"

views_col_letter = "E"
country_col_letter = "P"

for col in range(1, ws_records.max_column + 1):
    val = ws_records.cell(row=1, column=col).value
    if val == "views":
        views_col_letter = get_column_letter(col)
    elif val == "country":
        country_col_letter = get_column_letter(col)

ws_summary["A2"] = "Total Combined Views"
ws_summary["B2"] = f'=SUM(Records!{views_col_letter}:{views_col_letter})'

ws_summary["A3"] = "Overall Average Views"
ws_summary["B3"] = f'=AVERAGE(Records!{views_col_letter}:{views_col_letter})'

ws_summary["A4"] = "US Records Count"
ws_summary["B4"] = f'=COUNTIF(Records!{country_col_letter}:{country_col_letter}, "US")'

ws_summary["A5"] = "GB Records Count"
ws_summary["B5"] = f'=COUNTIF(Records!{country_col_letter}:{country_col_letter}, "GB")'

ws_summary["A6"] = "IN Records Count"
ws_summary["B6"] = f'=COUNTIF(Records!{country_col_letter}:{country_col_letter}, "IN")'

# --- Requirement (a): Pivot Table / Summary Sheet ---
ws_pivot = wb.create_sheet(title="Pivot_Summary")
df_records["category_id"] = df_records["category_id"].astype(str)
df_cat["category_id"] = df_cat["category_id"].astype(str)

pivot_df = df_records.groupby(["category_id", "country"])["views"].mean().unstack().fillna(0)
pivot_df = pivot_df.merge(df_cat, on="category_id", how="left").set_index("category_name")[["US", "GB", "IN"]]

ws_pivot["A1"] = "Category Name"
ws_pivot["B1"] = "Avg Views (US)"
ws_pivot["C1"] = "Avg Views (GB)"
ws_pivot["D1"] = "Avg Views (IN)"

r_idx = 2
for cat_name, row in pivot_df.iterrows():
    ws_pivot.cell(row=r_idx, column=1, value=cat_name)
    ws_pivot.cell(row=r_idx, column=2, value=round(row.get("US", 0), 2))
    ws_pivot.cell(row=r_idx, column=3, value=round(row.get("GB", 0), 2))
    ws_pivot.cell(row=r_idx, column=4, value=round(row.get("IN", 0), 2))
    r_idx += 1

# --- Requirement (d): Enable AutoFilters ---
ws_records.auto_filter.ref = ws_records.dimensions

# --- Requirement (e): Conditional Formatting (Top 10% Views) ---
green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
green_font = Font(color="006100")
top_10_percent_rule = CellIsRule(operator='greaterThan', formula=['PERCENTILE(Records!E2:E1000, 0.9)'], stopIfTrue=True, fill=green_fill, font=green_font)
ws_records.conditional_formatting.add(f"{views_col_letter}2:{views_col_letter}{ws_records.max_row}", top_10_percent_rule)

# Save workbook
wb.save(excel_path)
print(f"✅ Excel workbook successfully created and saved at: {excel_path}")

✅ Excel workbook successfully created and saved at: trendpulse/part4_sql_database/outputs/TrendPulse_Analysis.xlsx
